## Parallel Workflow 

In [31]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import os
import operator
from dotenv import load_dotenv
from langchain_groq import ChatGroq

In [20]:
load_dotenv()

model = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant",
    temperature=0.7
)

In [21]:
class EvaluationScheme(BaseModel):
    feedback: str = Field(description="Detailed feedback on the essay's language quality.")
    score: int = Field(description="A score out of 10 evaluating the essay's language quality.", ge=0, le=10)
    

In [23]:
structured_model=model.with_structured_output(EvaluationScheme)
    
essay = """
    In the heart of the bustling city, there was a small park that offered a serene escape from the chaos. The park was filled with vibrant flowers, towering trees, and a tranquil pond that reflected the clear blue sky. People from all walks of life would come to the park to relax, read a book, or simply enjoy the beauty of nature. The sound of birds chirping and leaves rustling in the gentle breeze created a soothing atmosphere that made it a perfect place for contemplation and rejuvenation.
    """

prompt = f"""
        Evaluate the language quality of the following essay.
        Return output strictly in JSON format:
        {{
            "feedback":"your detailed feedback",
            "score": integer score out of 10
        }}
        
Essay :{essay}"""

structured_model.invoke(prompt)

EvaluationScheme(feedback="The essay is well-written, with vivid descriptions of the park's scenery. The use of sensory details, such as the sound of birds chirping and leaves rustling, effectively transports the reader to the scene. The language is simple yet effective, making it accessible to a wide range of readers. However, the essay could benefit from more depth and complexity in its ideas and analysis. Overall, it is a pleasant and calming piece of writing.", score=8)

In [32]:
class EssayState(TypedDict):
    essay:str
    cot_feedback:str
    doa_feedback:str
    language_feedback:str
    overall_feedback:str
    individual_score:Annotated[list[int],operator.add]
    final_avg_score:float


In [33]:
def evaluate_language_feedback(state:EssayState) -> EssayState:
    essay = state['essay']
    prompt=f'Evaluate the language quality of the following essay and provide feedback and give a score out of 10 \n: {essay}'
    output=structured_model.invoke(prompt)
    return {'language_feedback':output.feedback, 'individual_score':[output.score]}
    
def evaluate_analysis_feedback(state:EssayState) -> EssayState:
    essay = state['essay']
    prompt=f'Evaluate the depth of analysis of the following essay and provide feedback and give a score out of 10 \n: {essay}'
    output=structured_model.invoke(prompt)
    return {'doa_feedback':output.feedback, 'individual_score':[output.score]}

def evaluate_thought_feedback(state:EssayState) -> EssayState:
    essay = state['essay']
    prompt=f'Evaluate the clarity of thought of the following essay and provide feedback and give a score out of 10 \n: {essay}'
    output=structured_model.invoke(prompt)
    return {'cot_feedback':output.feedback, 'individual_score':[output.score]}

def final_evaluation(state:EssayState) -> EssayState:
    # summary
    prompt=f'Based on the following feedback, provide an overall evaluation of the essay and give an overall score out of 10:\n\nLanguage Feedback: {state["language_feedback"]}\n\nDepth of Analysis Feedback: {state["doa_feedback"]}\n\nClarity of Thought Feedback: {state["cot_feedback"]}'
    overall_feedback = model.invoke(prompt).content
    # average score
    avg_score = sum(state['individual_score'])/len(state['individual_score'])
    return {'overall_feedback':overall_feedback, 'final_avg_score':avg_score}


In [34]:
graph = StateGraph(EssayState)

# add nodes
graph.add_node("evaluate_language_feedback",evaluate_language_feedback)
graph.add_node("evaluate_analysis_feedback",evaluate_analysis_feedback)
graph.add_node("evaluate_thought_feedback",evaluate_thought_feedback)
graph.add_node("final_evaluation",final_evaluation)

# add edges
graph.add_edge(START,'evaluate_language_feedback')
graph.add_edge(START,'evaluate_analysis_feedback')
graph.add_edge(START,'evaluate_thought_feedback')

graph.add_edge('evaluate_language_feedback','final_evaluation')
graph.add_edge('evaluate_analysis_feedback','final_evaluation')
graph.add_edge('evaluate_thought_feedback','final_evaluation')

graph.add_edge('final_evaluation',END)

In [35]:
# compile
workflow = graph.compile()

# execute
initial_state = {'essay':essay}
final_state = workflow.invoke(initial_state)
print(final_state)

{'essay': '\n    In the heart of the bustling city, there was a small park that offered a serene escape from the chaos. The park was filled with vibrant flowers, towering trees, and a tranquil pond that reflected the clear blue sky. People from all walks of life would come to the park to relax, read a book, or simply enjoy the beauty of nature. The sound of birds chirping and leaves rustling in the gentle breeze created a soothing atmosphere that made it a perfect place for contemplation and rejuvenation.\n    ', 'cot_feedback': "The essay has a clear and concise description of the park's serene atmosphere, effectively conveying a sense of tranquility. However, it could benefit from a more developed thesis statement and a more nuanced exploration of the park's significance. The writing is descriptive and engaging, but may benefit from a stronger focus on the author's thoughts and reflections.", 'doa_feedback': 'The analysis provided in this essay is shallow, focusing mainly on the phys